# Docker Basics -- Soft Prerequisite Refresher

**Who is this for?** You are already running this notebook inside a Docker container. This notebook helps you understand what that means and gives you the basics to inspect, debug, and reason about containerized environments. Session 30 covers Docker in depth for ML deployment; this is just the groundwork.

**Estimated time:** 30--45 minutes.

**What it covers:**
1. What is Docker and why it matters
2. Containers vs VMs
3. Key concepts: images, containers, volumes, ports
4. Dockerfile basics
5. Docker Compose basics
6. Essential commands
7. Volumes and data persistence
8. Common gotchas

If any section feels unfamiliar (not just rusty), check the references at the bottom.

---
## 1. What is Docker and Why It Matters

Docker packages an application and all its dependencies into a **container** -- a lightweight, isolated environment that runs the same way on any machine.

Why this matters for ML:
- **Reproducibility:** "It works on my machine" stops being a problem. Your training environment is defined in code.
- **Consistency:** Everyone in the course runs the same Python version, the same library versions, the same OS-level dependencies.
- **Isolation:** Your project's dependencies don't conflict with your system's packages.

Right now, your Jupyter notebook server is running inside a Docker container built from the course's `Dockerfile`. The Python version, the installed libraries (PyTorch, scikit-learn, etc.) -- all of that is defined in a few lines of configuration.

Let's verify we are indeed inside a container:

In [ ]:
# The /.dockerenv file exists inside Docker containers
import os
print("Inside a container:", os.path.exists("/.dockerenv"))
print("Hostname:", os.uname().nodename)  # containers get a random hostname (the container ID)

---
## 2. Containers vs Virtual Machines

Both provide isolation, but they work at different levels:

```
  Virtual Machine                    Container
  +-----------------+               +-----------------+
  |   Application   |               |   Application   |
  +-----------------+               +-----------------+
  |   Libraries     |               |   Libraries     |
  +-----------------+               +-----------------+
  |   Guest OS      |               |  (no guest OS)  |
  +-----------------+               +-----------------+
  |   Hypervisor    |               |  Container      |
  |                 |               |  Runtime        |
  +-----------------+               +-----------------+
  |   Host OS       |               |   Host OS       |
  +-----------------+               +-----------------+
  |   Hardware      |               |   Hardware      |
  +-----------------+               +-----------------+
```

**VMs** run a full guest operating system on top of a hypervisor. Heavy (GBs of RAM), slow to start (minutes).

**Containers** share the host OS kernel. They isolate processes using Linux namespaces and cgroups. Light (MBs), fast to start (seconds).

For ML work, containers are almost always the right choice. You rarely need a full VM just to run a training script.

---
## 3. Key Concepts: Images, Containers, Volumes, Ports

Four terms you need to know:

| Concept | What it is | Analogy |
|---------|-----------|--------|
| **Image** | A read-only template with the OS, libraries, and app code. Built from a Dockerfile. | A class definition |
| **Container** | A running instance of an image. Has its own filesystem, network, and process space. | An object (instance of a class) |
| **Volume** | A mechanism to persist data outside the container's filesystem. Survives container restarts. | A USB drive plugged into the container |
| **Port mapping** | Connects a port on your host machine to a port inside the container. | A door between your machine and the container |

You can create many containers from the same image. Each container is independent.

### Exercise 3.1

Think about the course setup. Answer these questions (no code needed, just think):

1. When you ran `docker compose up` to start this notebook, did Docker create an image, a container, or both?
2. The notebook is accessible at `http://localhost:8888`. Which port mapping makes this possible?
3. If you `pip install some-package` right now, will that package survive a `docker compose down && docker compose up`? Why or why not?

---
## 4. Dockerfile Basics

A `Dockerfile` is a recipe for building an image. Each instruction creates a **layer** in the image.

The four instructions you see most often:

| Instruction | Purpose | Example |
|-------------|---------|--------|
| `FROM` | Base image to start from | `FROM python:3.14-slim` |
| `RUN` | Execute a command during build (install packages, etc.) | `RUN pip install numpy` |
| `COPY` | Copy files from your machine into the image | `COPY requirements.txt /app/` |
| `CMD` | Default command when the container starts | `CMD ["python", "app.py"]` |

Other useful instructions: `WORKDIR` (set working directory), `EXPOSE` (document which port the app uses), `ENV` (set environment variables).

Here is the **actual Dockerfile** for this course:

```dockerfile
FROM python:3.14-slim

WORKDIR /workspace

RUN pip install --no-cache-dir \
    jupyter \
    numpy \
    pandas \
    matplotlib \
    scikit-learn \
    xgboost \
    torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cpu

EXPOSE 8888

CMD ["jupyter", "notebook", "--ip=0.0.0.0", "--port=8888", "--no-browser", "--allow-root", "--NotebookApp.token=''"]
```

Reading it line by line:
1. **`FROM python:3.14-slim`** -- Start from a minimal Python 3.14 image (Debian-based, small footprint).
2. **`WORKDIR /workspace`** -- All subsequent commands run in `/workspace`.
3. **`RUN pip install ...`** -- Install all the ML libraries we need. `--no-cache-dir` keeps the image smaller.
4. **`EXPOSE 8888`** -- Documents that the container listens on port 8888 (does not actually publish it -- that's done in docker-compose.yml).
5. **`CMD [...]`** -- When the container starts, it launches Jupyter Notebook bound to all interfaces (`0.0.0.0`), with no auth token.

### Exercise 4.1

Open the course's `Dockerfile` (it is at the root of the project, which is mounted at `/workspace/Dockerfile` inside this container). Read it and answer:

1. What base image does it use? What does the `-slim` suffix mean?
2. Why is `--ip=0.0.0.0` needed in the CMD? (Hint: what happens if Jupyter binds only to `127.0.0.1` inside a container?)
3. If you wanted to add `seaborn` to the course environment, which line would you modify?

In [ ]:
# Read the course Dockerfile from inside the container
!cat /workspace/Dockerfile

---
## 5. Docker Compose Basics

`docker compose` is a tool for defining and running multi-container applications using a YAML file. Even for single-container setups (like ours), it is convenient because it captures all the run options (ports, volumes, build context) in a file instead of a long `docker run` command.

Here is the **actual `docker-compose.yml`** for this course:

```yaml
services:
  jupyter:
    build: .
    ports:
      - "8888:8888"
    volumes:
      - .:/workspace
```

Breaking it down:
- **`services.jupyter`** -- Defines a service named `jupyter`.
- **`build: .`** -- Build the image from the Dockerfile in the current directory.
- **`ports: "8888:8888"`** -- Map host port 8888 to container port 8888. Format is `host:container`.
- **`volumes: .:/workspace`** -- Mount the current directory (`.`) on the host to `/workspace` inside the container. This is a **bind mount**.

The bind mount is critical: it means the files on your host machine and the files inside the container at `/workspace` are the **same files**. When you save a notebook, it saves to your host filesystem. When you `docker compose down`, your notebooks are still there.

### Exercise 5.1

Read the course's `docker-compose.yml`:

In [ ]:
!cat /workspace/docker-compose.yml

Answer these:
1. If you changed the ports line to `"9999:8888"`, what URL would you use to access Jupyter?
2. If you removed the `volumes` section entirely, what would happen to notebooks you create?

---
## 6. Essential Commands

These are the commands you will use regularly. Run them from a terminal on your **host machine** (not from inside the container). Here we use `!` to run them from within the notebook, but note that some commands (like `docker ps`) need access to the Docker daemon, which may or may not be available from inside the container.

| Command | What it does |
|---------|-------------|
| `docker ps` | List running containers |
| `docker ps -a` | List all containers (including stopped) |
| `docker images` | List locally available images |
| `docker compose up` | Start services defined in docker-compose.yml |
| `docker compose up -d` | Same, but in the background (detached) |
| `docker compose down` | Stop and remove containers |
| `docker compose build` | Rebuild images (after Dockerfile changes) |
| `docker exec -it <container> bash` | Open a shell inside a running container |
| `docker logs <container>` | View container output/logs |

### Exercise 6.1 -- Inspect Your Running Container

Open a **terminal on your host machine** (not this notebook) and run the following commands. Write down what you observe.

```bash
# 1. List running containers -- find yours
docker ps

# 2. Note the CONTAINER ID and IMAGE name

# 3. Open a shell inside your running container
docker exec -it <container_id> bash

# 4. Inside the container, run:
python --version
pip list | head -20
ls /workspace
exit
```

Questions:
1. What is the container ID and image name?
2. Does `ls /workspace` show the same files you see in Jupyter's file browser?
3. What Python version is installed?

### Exercise 6.2 -- Explore Images

On your host machine, run:

```bash
docker images
```

Find the course image. How large is it? Why is it that size? (Hint: PyTorch is not small.)

We can verify some things from inside the container too:

In [ ]:
%%bash
echo "=== Python version ==="
python --version

echo ""
echo "=== Working directory ==="
pwd

echo ""
echo "=== Files in /workspace (first 15) ==="
ls /workspace | head -15

echo ""
echo "=== Key installed packages ==="
pip list 2>/dev/null | grep -iE "torch|numpy|pandas|scikit|jupyter|xgboost"

---
## 7. Volumes and Data Persistence

This is the most important practical concept for day-to-day Docker use.

A container's filesystem is **ephemeral** -- when the container is removed, everything inside it is gone. Volumes solve this by mapping a directory on your host to a directory inside the container.

In our `docker-compose.yml`:
```yaml
volumes:
  - .:/workspace
```

This means:
- The course directory on your host is mounted at `/workspace` in the container.
- Files you create/edit in `/workspace` are saved to your host.
- Files you create **outside** `/workspace` (e.g., in `/tmp` or `/root`) exist only inside the container and vanish when it is removed.

Let's demonstrate:

In [ ]:
%%bash
# This file is in the mounted volume -- it persists
echo "I will survive" > /workspace/test_persist.txt

# This file is NOT in the volume -- it will be lost when the container is removed
echo "I will vanish" > /tmp/test_ephemeral.txt

echo "Both files exist right now:"
cat /workspace/test_persist.txt
cat /tmp/test_ephemeral.txt

### Exercise 7.1

Try this experiment on your host machine:

1. Run `docker compose down` to stop the container.
2. Check that `test_persist.txt` exists in your course directory on the host.
3. Run `docker compose up -d` to start it again.
4. Open a shell in the new container: `docker exec -it <container_id> bash`
5. Check: does `/workspace/test_persist.txt` exist? Does `/tmp/test_ephemeral.txt` exist?

This is the fundamental lesson: **only data in mounted volumes survives container recreation**.

---
## 8. Common Gotchas

**1. Container state is ephemeral.**
If you `pip install` a package inside a running container, it works until you recreate the container (`docker compose down && up`). To make it permanent, add it to the `Dockerfile` and rebuild.

**2. Data outside volumes is lost.**
Trained models, datasets, logs -- if they are saved outside the mounted volume, they disappear. Always save important files to the mounted directory.

**3. Port conflicts.**
If another process on your host is using port 8888, `docker compose up` will fail. Either stop the other process or change the host port in `docker-compose.yml`.

**4. Image cache.**
Docker caches image layers. If you change the `Dockerfile`, you need to run `docker compose build` (or `docker compose up --build`) to rebuild. Otherwise, Docker will use the cached (old) image.

**5. File permissions.**
The container often runs as root. Files created inside the container in the mounted volume may be owned by root on your host. This can cause permission issues. (The course setup avoids this by running Jupyter with `--allow-root`, but be aware of it in other projects.)

### Exercise 8.1

Scenario debugging. For each situation, explain what went wrong and how to fix it:

1. You installed `seaborn` with `pip install seaborn` inside the container. It worked fine all week. On Monday you ran `docker compose down && docker compose up` and now `import seaborn` fails.

2. You trained a model and saved it to `/root/my_model.pt`. After restarting the container, the file is gone.

3. You modified the `Dockerfile` to add `scipy`, ran `docker compose up`, but `import scipy` still fails.

---
## Self-Assessment

You should be able to answer these without looking anything up:

1. **What is the difference between a Docker image and a container?**

2. **You run `docker compose down`. Which of these survive: (a) a notebook saved in `/workspace`, (b) a Python package you pip-installed, (c) a file you saved in `/tmp` inside the container?**

3. **What does the `volumes: - .:/workspace` line in docker-compose.yml do, and why is it essential for the course setup?**

4. **You changed the Dockerfile. What command do you run to apply the changes?**

If any of these gave you trouble, re-read the relevant section. Session 30 will go much deeper (multi-stage builds, GPU containers, pushing to registries), but these basics should be solid before then.

---
## References

- [Docker Get Started Guide](https://docs.docker.com/get-started/) -- Official tutorial. Parts 1--3 cover everything in this notebook.
- [Docker Compose Overview](https://docs.docker.com/compose/) -- Official docs on Compose, including the file format reference.
- [Best Practices for Writing Dockerfiles](https://docs.docker.com/develop/develop-images/dockerfile_best-practices/) -- Layer caching, multi-stage builds, minimizing image size.

In [ ]:
# Cleanup
import os
if os.path.exists("/workspace/test_persist.txt"):
    os.remove("/workspace/test_persist.txt")
    print("Cleaned up test_persist.txt")